In [ ]:
# Setting up the notebook
import sys
sys.path.append("../scripts")

In [ ]:
# Import statements
import pandas as pd
import matplotlib.pyplot as plt

from ipywidgets import interact, Dropdown

from config import DATA_DIR, REQUESTS_DIR, RESPONSES_DIR, SCRIPTS_DIR
import functions as func

# 1. PRE-PROCESSING

## 1.1 Creation of Data Table

In [ ]:
requests_df = func.create_df_of_requests(REQUESTS_DIR)
responses_df = func.create_df_of_responses(RESPONSES_DIR)
joined_df = func.join_requests_and_responses(requests_df, responses_df)

In [ ]:
# A result
joined_df.head()

## 1.2 Solving Pre-Requisites

### Defining unique location IDs by clustering: DBSCAN

**PURPOSE**

The coordinates provided are floats (created by transformation of the original 'real' coordinates). Analyses requiring comparison of unique locations necessitate assignment of a unique location ID. We posited that small differences in floating point numbers might in reality refer to the same location. A clustering function was developed to assign locations that are tightly spread within a defined cut-off (e.g., 5 m) to the same location ID.

In [ ]:
# Maxime to either add functions to the "functions.py" file or create a new file for the location IDs + verification; then import for execution.


### Finding time and task tresholds ("cut-off") for further analyses

**PURPOSE**

Time: requests follow a pulsed pattern, with a first wave of intense activity representing organization of the daily deliveries, which is our subject of interest. A second large activity peak is irrelevant for the analysis. We want to define where to set the "cursor". 

Tasks: some trips contain a lot of tasks, some trips contain only a few tasks. We are not interested in the trips with a few tasks. Find the cutoff. 

#### Time

In [ ]:
# read inputfile excel - to run functions 
df, info= func.read_inputfile(DATA_DIR / "ModifiedQueryRows.xlsx")

In [ ]:
func.define_cutoff_time(df, depot="0521")

#### Tasks

In [ ]:
summary, info = func.define_cutoff_stops(df, depot="0521")

In [ ]:
# Choice of time threshold
thresh = pd.to_datetime("11:00:00").time()

# 2. DATA EXPLORATION

## To start 
Create filtered and annotated dfs 

In [ ]:
#Filter on time and tasks cutoffs (11h and 30 min tasks)
filtered_time_and_tasks_df= func.filter_on_cutoff_time_and_tasks(df)

#Annotate with request sequence numbers to compare first and consecutive requests within a trip
annotated_with_request_seq = func.annotate_requests_with_sequence_num(filtered_time_and_tasks_df)

# Create a table with 1 row per trip (route/day)
aggregated_trips_df = func.aggregate_trips(filtered_time_and_tasks_df)


In [ ]:
func.trips_per_day_plot(filtered_df= filtered_time_and_tasks_df)

In [ ]:
route_summary = func.summarize_requests(trip_df=aggregated_trips_df, show_plots=True)

In [ ]:
aggregate_route_df = func.summarize_type_of_requests(trip_df = aggregated_trips_df)


In [ ]:
trip_task_var_df, summary_trip_var_df = func.task_variability_within_trips(filtered_time_and_tasks_df)

In [ ]:
func.plot_task_variability_in_trips(trip_task_var_df)

# 3. DATA ANALYSIS

## 3.1 GRANULAR ANALYSIS

**RESEARCH QUESTION**

Two optimization models are used in combination: model 1 assigns tasks to routeIDs (regions), model 2 predicts the optimal sequence of locations (the optimal "trip"). Our objective is to evaluate the most common change made by the user: a change in the number of taskIDs (which would suggest non-optimality of model 1) or a change in the sequence of taskIDs (locations) (which would suggest non-optimality of model 2).

In [ ]:
responses_dir = RESPONSES_DIR
result_df = func.compute_levenshtein_from_df(annotated_with_request_seq, responses_dir=responses_dir)

routes = sorted(result_df["route_id"].unique().tolist())

interact(
    lambda route_id: func.plot_route_grouped_bars(result_df, route_id=route_id),
    route_id=Dropdown(options=routes, value=routes[0] if routes else None, description="route_id")
)

In [ ]:
func.plot_distribution_levensthein(result_df)

## 3.2 CHANGE IMPACT ANALYSIS

**RESEARCH QUESTION**

The project's base hypothesis states that the model predictions are deemed non-optimal given the extent of human intervention observed to yield final routes. The trueness of this hypothesis deserves further assessment: does human intervention produce a more optimal result?

For this, the total distance to be driven following the model's recommendations can be compared to the distance to be driven after all human interventions. As taskIDs are not constant within regions and days (i.e., tasks are assumed to migrate between regions), an aggregated computation across all regions connected to warehouse 0521 will be performed. This is based on the assumption that taskIDs may change within routeIDs (per day) due to assignment to other routeIDs of the same warehouse. 

*STEP 1: Evaluation of Assumption*

In [ ]:
filtered_requests_df = requests_df[requests_df["time"] < thresh].loc[:, ["route_id", "date", "time", "request_type", "location_id"]]
counted_tasks_df = func.count_tasks_per_sequence(filtered_requests_df)
select_compare_start_end_df = func.select_compare_start_and_end(counted_tasks_df)

In [ ]:
# Verifying the assumption
plt.style.use(SCRIPTS_DIR / "plot_style.mplstyle") # May have to be put elsewhere

ax = (select_compare_start_end_df.groupby("date")["abs_diff_end_start"]
    .sum()
    .reset_index(name="diff_per_day")
    .plot(x="date", y="diff_per_day", kind="bar", legend=False))

ax.set_title("Change in number of tasks over warehouse 0521 between start and end of shift")

Question: if the tasks do not stay in depot 0521, do they go to another depot?
Analysis on Excel with all depots. 
First, read Excel without setting depot number. 

In [ ]:
df_all_depots, info= func.read_inputfile(DATA_DIR / "ModifiedQueryRows.xlsx", depot = None)

#Needed dfs for analysis
filtered_time_and_tasks_df_all= func.filter_on_cutoff_time_and_tasks(df_all_depots)
trip_task_var_df_all, summary_trip_var_df_all = func.task_variability_within_trips(filtered_time_and_tasks_df_all, show_plots = False)

In [ ]:
delta_df = func.delta_n_tasks_first_last_req_per_day(trip_task_var_df_all)

**_CONCLUSION_**

The assumption is clearly false. Further comparison of the total distance between model output and human output is not useful.